# RoBacTutor — RAG Component
### MSc Computer Science Dissertation — University College Birmingham
**Author:** Vasile Bria | **Student ID:** BRI23222497  
**Supervisor:** Farah Shahid  
**Component:** Retrieval-Augmented Generation (RAG)  
**Embeddings:** all-MiniLM-L6-v2 (Sentence-BERT)  
**Index:** FAISS flat L2

---
## Instructions
1. Set runtime to **T4 GPU + High-RAM**
2. Run Cell 1 (install) — wait for restart
3. Run cells 2 through 7 in order
4. Upload your 20 ANCE PDF files when prompted in Cell 3
5. Upload your LoRA adapter zip when prompted in Cell 5

---
**UPDATED:** Cells 3-4 now also process curriculum textbook manuals alongside the original 20 ANCE exam/barem PDFs, tagged separately via `source_type` ('exam' vs 'manual'). This tests whether textbook content fixes the conceptual-query retrieval gap diagnosed in Dissertation Table 4.7 (e.g. 'ce este elenismul?' retrieving nothing useful from exam-only content).

## Cell 1 — Install
> ⚠️ Runtime restarts after this. Skip after restart.

In [ ]:
# RoBacTutor RAG — Cell 1: Install
# Vasile Bria | BRI23222497 | UCB 2025-2026

!pip install -q \
    "transformers==4.46.0" \
    "bitsandbytes>=0.46.1" \
    "peft==0.12.0" \
    "accelerate==0.34.2" \
    "sentence-transformers==2.7.0" \
    "faiss-gpu" \
    "pdfplumber" \
    "numpy==1.26.4" \
    "sentencepiece" \
    "fsspec==2024.6.1"

print('All packages installed!')
print('Restarting...')
import os
os.kill(os.getpid(), 9)


## Cell 2 — Verify Environment

In [ ]:
# RoBacTutor RAG — Cell 2: Verify
# Vasile Bria | BRI23222497 | UCB 2025-2026

import torch
import sentence_transformers
import faiss
import pdfplumber

print('=' * 50)
print('RoBacTutor RAG Pipeline')
print('Vasile Bria | BRI23222497 | UCB')
print('=' * 50)
print(f'PyTorch:             {torch.__version__}')
print(f'sentence-transformers: {sentence_transformers.__version__}')
print(f'FAISS:               {faiss.__version__}')
print(f'CUDA:                {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:                 {torch.cuda.get_device_name(0)}')
print('\n✅ Environment ready')


## Cell 3 — Upload ANCE PDFs and Extract Text
> Upload all 20 ANCE PDF files when prompted.
> Extraction takes 2-3 minutes.

In [ ]:
# RoBacTutor RAG — Cell 3: Extract text from ANCE exam PDFs AND textbook manuals
# Vasile Bria | BRI23222497 | UCB 2025-2026
#
# UPDATED (v3): textbooks are now read from Google Drive instead of the
# browser upload widget. The 562MB textbook upload was the single slowest,
# most fragile step in this whole notebook — every runtime restart meant
# re-uploading all of it from scratch. Mounting Drive once means this never
# needs to happen again, regardless of how many times the session restarts.
#
# Exam PDFs stay as a normal upload — they're small (a few hundred KB each)
# and fast, no need to change that part.
#
# SETUP (one-time): upload your 11 textbook PDFs to a folder in your Google
# Drive first, e.g. "MyDrive/robactutor_manuale/", then set DRIVE_MANUALE_DIR
# below to match.
#
# Extraction uses PyMuPDF (see v2 notes below) — ~10x more memory-efficient
# than pdfplumber, no subprocess isolation needed.

from google.colab import files, drive
import re, glob, os

!pip install -q pymupdf
import fitz

DRIVE_MANUALE_DIR = '/content/drive/MyDrive/robactutor_manuale'

def clean_text(text):
    text = re.sub(r'\(cid:\d+\)', '', text)   # harmless no-op safety net; PyMuPDF doesn't produce these
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

def extract_text_from_pdf(filepath):
    doc = fitz.open(filepath)
    text = ''
    for page in doc:
        text += page.get_text() + '\n'
    doc.close()
    return clean_text(text)

def detect_subject(filename):
    # FIXED: History files are titled "Istoria romanilor si universala",
    # which contains the substring "rom" -- checking Romanian before History
    # silently mistagged every History textbook as limba_romana, making
    # them invisible to any query filtered by subject=='istorie'. Checking
    # 'ist' before 'rom' fixes this.
    fn = filename.lower()
    if 'mat' in fn: return 'matematica'
    if 'eng' in fn or 'engl' in fn: return 'limba_engleza'
    if 'ist' in fn or 'hist' in fn: return 'istorie'
    if 'rom' in fn or 'lim' in fn: return 'limba_romana'
    return 'general'

def process_uploaded_batch(uploaded_dict, source_type):
    docs = []
    for filename, content in uploaded_dict.items():
        with open(filename, 'wb') as f:
            f.write(content)
        text = extract_text_from_pdf(filename)
        subject = detect_subject(filename)
        year = '2024' if '2024' in filename else '2025' if '2025' in filename else 'unknown'
        docs.append({
            'filename': filename, 'subject': subject, 'year': year,
            'text': text, 'source_type': source_type,
        })
        print(f'  \u2705 {filename} ({subject}, {year}, {source_type}) \u2014 {len(text)} chars')
    return docs

def process_drive_batch(directory, source_type):
    docs = []
    filepaths = sorted(glob.glob(os.path.join(directory, '*.pdf')))
    if not filepaths:
        print(f'  \u26a0\ufe0f  No PDFs found in {directory} \u2014 check the folder path and that Drive is mounted.')
    for filepath in filepaths:
        filename = os.path.basename(filepath)
        if 'SCANNED' in filename:
            print(f'  \u23ed\ufe0f  Skipping {filename} (marked scanned/image-only)')
            continue
        text = extract_text_from_pdf(filepath)
        subject = detect_subject(filename)
        year = '2024' if '2024' in filename else '2025' if '2025' in filename else 'unknown'
        docs.append({
            'filename': filename, 'subject': subject, 'year': year,
            'text': text, 'source_type': source_type,
        })
        print(f'  \u2705 {filename} ({subject}, {year}, {source_type}) \u2014 {len(text)} chars')
    return docs

print('=== Upload your 20 ANCE exam/barem PDFs ===')
uploaded_exam = files.upload()
exam_docs = process_uploaded_batch(uploaded_exam, source_type='exam')

print()
print('=== Mounting Google Drive for textbook manuals ===')
drive.mount('/content/drive')
manual_docs = process_drive_batch(DRIVE_MANUALE_DIR, source_type='manual')

documents = exam_docs + manual_docs
print(f'\nTotal documents: {len(documents)} ({len(exam_docs)} exam, {len(manual_docs)} manual)')
print('\u2705 Text extraction complete')


## Cell 4 — Chunk Text and Build FAISS Index
> Splits documents into ~200 word passages and embeds them.
> Takes 3-5 minutes.

In [ ]:
# RoBacTutor RAG — Cell 4: Chunk and embed
# Vasile Bria | BRI23222497 | UCB 2025-2026
#
# UPDATED: chunk_metadata now carries 'source_type' ('exam' or 'manual')
# through from each document, so retrieval behaviour can be compared by
# source type later (e.g. "did this answer get grounded in a barem or a
# textbook explanation?").

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import re

CHUNK_SIZE = 200
CHUNK_OVERLAP = 40
EMBED_MODEL = 'all-MiniLM-L6-v2'

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = ' '.join(words[start:end])
        if len(chunk.strip()) > 50:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

all_chunks = []
chunk_metadata = []

for doc in documents:
    chunks = chunk_text(doc['text'])
    for chunk in chunks:
        all_chunks.append(chunk)
        chunk_metadata.append({
            'filename': doc['filename'],
            'subject': doc['subject'],
            'year': doc['year'],
            'source_type': doc['source_type'],
            'text': chunk
        })

print(f'Total chunks: {len(all_chunks)}')
print(f'  From exam PDFs: {sum(1 for c in chunk_metadata if c["source_type"] == "exam")}')
print(f'  From textbooks: {sum(1 for c in chunk_metadata if c["source_type"] == "manual")}')
print(f'Avg chunk length: {np.mean([len(c.split()) for c in all_chunks]):.0f} words')

print(f'\nLoading embedding model: {EMBED_MODEL}...')
embedder = SentenceTransformer(EMBED_MODEL)

print('Generating embeddings...')
embeddings = embedder.encode(
    all_chunks, batch_size=64, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
)

print('Building FAISS index...')
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings.astype('float32'))

print(f'\n\u2705 FAISS index built!')
print(f'   Embedding dimension: {dimension}')
print(f'   Total vectors: {index.ntotal}')


## Cell 5 — Load Fine-Tuned RoBacTutor Model
> Upload your `robactutor_lora_adapters_BRI23222497.zip` when prompted.

In [ ]:
# RoBacTutor RAG — Cell 5: Load fine-tuned model
# Vasile Bria | BRI23222497 | UCB 2025-2026

from google.colab import files
import zipfile, os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Upload adapter zip
print('Upload robactutor_lora_adapters_BRI23222497.zip...')
uploaded = files.upload()
zip_file = list(uploaded.keys())[0]

# Extract
ADAPTER_DIR = './robactutor-adapters'
os.makedirs(ADAPTER_DIR, exist_ok=True)
with zipfile.ZipFile(zip_file, 'r') as zf:
    zf.extractall(ADAPTER_DIR)
print(f'Extracted to {ADAPTER_DIR}')
print('Files:', os.listdir(ADAPTER_DIR))

# Load base model in 4-bit
MODEL_ID = 'OpenLLM-Ro/RoMistral-7B-Instruct'
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

print('\nLoading base model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)

# Load LoRA adapters
print('Loading LoRA adapters...')
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

used  = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\n✅ RoBacTutor model loaded!')
print(f'   VRAM used: {used:.1f} / {total:.1f} GB')


## Cell 6 — RAG Pipeline

In [ ]:
# RoBacTutor RAG — Cell 6: RAG pipeline
# Vasile Bria | BRI23222497 | UCB 2025-2026

import torch

TOP_K = 3  # number of chunks to retrieve

def retrieve(query, subject=None, top_k=TOP_K):
    """Retrieve top-k relevant chunks for a query."""
    # Embed the query
    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype('float32')

    # Search FAISS index
    scores, indices = index.search(query_embedding, top_k * 3)

    # Filter by subject if provided
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0: continue
        meta = chunk_metadata[idx]
        if subject and meta['subject'] != subject:
            continue
        results.append({
            'text': meta['text'],
            'subject': meta['subject'],
            'year': meta['year'],
            'score': float(score)
        })
        if len(results) >= top_k:
            break

    return results

def generate_with_rag(instruction, subject=None, max_new_tokens=400):
    """Generate a response using RAG-augmented context."""

    # Step 1: Retrieve relevant context
    retrieved = retrieve(instruction, subject=subject)

    # Step 2: Build context string
    context = ''
    if retrieved:
        context = 'Context din materialele oficiale ANCE:\n'
        for i, r in enumerate(retrieved):
            context += f'[{i+1}] ({r["subject"]}, {r["year"]}): {r["text"][:300]}\n\n'

    # Step 3: Build prompt
    system = (
        'Esti RoBacTutor, un asistent educational specializat in pregatirea elevilor '
        'pentru examenul de Bacalaureat din Republica Moldova. '
        'Foloseste contextul furnizat pentru a oferi raspunsuri precise si bine fundamentate. '
        'Raspunzi in limba romana, cu exceptia intrebarilor de Limba Engleza.'
    )

    full_instruction = f'{context}{instruction}' if context else instruction
    prompt = f'<s>[INST] {system}\n\n{full_instruction} [/INST]'

    # Step 4: Generate
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return {
        'response': response,
        'retrieved_chunks': retrieved
    }

print('✅ RAG pipeline ready')
print(f'   Retrieval: top-{TOP_K} chunks from {index.ntotal} total')


## Cell 7 — Test RAG Pipeline

In [ ]:
# RoBacTutor RAG — Cell 7: Test RAG responses
# Vasile Bria | BRI23222497 | UCB 2025-2026
#
# UPDATED: max_new_tokens temporarily reduced to 200 (from the function
# default of 400) across both test loops, to leave more GPU memory headroom
# while diagnosing the retrieval-width fix on an 8GB/14.56GB card that has
# already hit one CUDA OOM this session. Raise back to 400 (or omit the
# argument entirely) once you've confirmed things are stable again.

test_questions = [
    ('Subiect: Matematica\n\nCalculati valoarea expresiei: (4/25)^1.5 * (-5)^3.', 'matematica'),
    ('Subiect: Limba engleza\n\nExplain the difference between present perfect and simple past.', 'limba_engleza'),
    ('Subiect: Istoria romanilor\n\nExplica termenul domnie in contextul Evului Mediu romanesc.', 'istorie'),
]

for i, (question, subject) in enumerate(test_questions):
    print(f'\n{"="*60}')
    print(f'Q{i+1}: {question[:70]}')
    print(f'Subject filter: {subject}')

    result = generate_with_rag(question, subject=subject, max_new_tokens=200)

    print(f'\nRetrieved {len(result["retrieved_chunks"])} chunks:')
    for j, chunk in enumerate(result['retrieved_chunks']):
        print(f'  [{j+1}] {chunk["subject"]} {chunk["year"]} (score: {chunk["score"]:.3f})')
        print(f'       {chunk["text"][:100]}...')

    print(f'\nResponse:')
    print(result['response'])


# --- Re-test the two queries that previously failed on the exam-only
# corpus (Dissertation Table 4.7), now that textbook content has been
# added and the retrieval search width has been widened (Cell 6). ---
diagnostic_questions = [
    ('ce este o functie?', 'matematica'),
    ('ce este elenismul?', 'istorie'),
]

print(f'\n{"#"*60}')
print('DIAGNOSTIC RE-TEST: conceptual queries after adding textbooks')
print(f'{"#"*60}')

for i, (question, subject) in enumerate(diagnostic_questions):
    print(f'\n{"="*60}')
    print(f'D{i+1}: {question}')
    print(f'Subject filter: {subject}')

    result = generate_with_rag(question, subject=subject, max_new_tokens=200)

    print(f'\nRetrieved {len(result["retrieved_chunks"])} chunks:')
    for j, chunk in enumerate(result['retrieved_chunks']):
        print(f'  [{j+1}] {chunk["subject"]} {chunk["year"]} (score: {chunk["score"]:.3f})')
        print(f'       {chunk["text"][:100]}...')

    print(f'\nResponse:')
    print(result['response'])


## Cell 8 — Save FAISS Index and Metadata

In [ ]:
# RoBacTutor RAG — Cell 8: Save index
# Vasile Bria | BRI23222497 | UCB 2025-2026

import faiss
import json
import zipfile
from google.colab import files

# Save FAISS index
faiss.write_index(index, 'robactutor_faiss.index')

# Save chunk metadata
with open('robactutor_chunks.json', 'w', encoding='utf-8') as f:
    json.dump(chunk_metadata, f, ensure_ascii=False, indent=2)

# Zip both
zip_name = 'robactutor_rag_BRI23222497.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write('robactutor_faiss.index')
    zf.write('robactutor_chunks.json')

print(f'Saved:')
print(f'  robactutor_faiss.index — FAISS vector index')
print(f'  robactutor_chunks.json — {len(chunk_metadata)} chunks with metadata')

files.download(zip_name)
print(f'\n✅ Downloaded: {zip_name}')
